In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/annotated_cpu/checkpoints/post_cell_18.pickle

trying: ['orig_output']
me:  2
trying: ['av_per_essay']
me:  15
trying: ['ax']
me:  15
trying: ['stopwords']
me:  0
trying: ['np']
me:  0
trying: ['all_gaps']
me:  35
trying: ['train_txt']
me:  1
trying: ['pd']
me:  0
trying: ['sample_submission']
me:  1
trying: ['glob']
me:  0
trying: ['test_txt']
me:  1
trying: ['sample_discourse_id']
me:  3
trying: ['cols_to_merge']
me:  25
trying: ['i_3']
me:  25
trying: ['train_first_last']
me:  19
trying: ['last_ones']
me:  25
trying: ['train_first']
me:  19
trying: ['train_last']
me:  19
trying: ['train']
me:  25
trying: ['txt_file']
me:  23
trying: ['data']
me:  23
trying: ['t']
me:  23
trying: ['myid']
me:  23
trying: ['myword']
me:  23
trying: ['cols_to_display']
me:  27
trying: ['mylen']
me:  23
trying: ['len_dict']


me:  23
trying: ['total_gaps']
me:  37
trying: ['word_dict']
me:  23
trying: ['IREWR_tmp2']
me:  33
trying: ['IREWR_plug_1']
me:  33
trying: ['tqdm']
me:  0
trying: ['warnings']
me:  0
trying: ['style']
me:  0
trying: ['i_1']
me:  21
trying: ['CountVectorizer']
me:  0
trying: ['counter']
me:  21
trying: ['plt']
me:  0
trying: ['FuncFormatter']
me:  0
trying: ['ax1']
me:  13
trying: ['ax2']
me:  13
trying: ['IREWR_plug_2']
me:  31
trying: ['IREWR_tmp']
me:  31
Declaring variable stopwords
Declaring variable np
Declaring variable pd
Declaring variable glob
Declaring variable tqdm
Declaring variable warnings
Declaring variable style
Declaring variable CountVectorizer
Declaring variable plt
Declaring variable FuncFormatter
Declaring variable train_txt
Declaring variable sample_submission
Declaring variable test_txt
Declaring variable orig_output
Declaring variable sample_discourse_id
Declaring variable ax1
Declaring variable ax2
Declaring variable av_per_essay
Declaring variable ax
Declari

In [3]:
%%RecordEvent
%%time
### cell 19 ###

def add_gap_rows(essay):
    cols = ['discourse_start', 'discourse_end', 'discourse_type', 'gap_length', 'gap_end_length']
    # filter once, avoid .query
    df_base = train.loc[train['id'] == essay, cols].reset_index(drop=True)
    print(df_base)
    if df_base.empty:
        return df_base
    # identify gap rows between existing segments (i >= 1)
    mask = df_base['gap_length'] > 0
    mask.iloc[0] = False
    gap_idxs = mask[mask].index
    if not gap_idxs.empty:
        starts = df_base['discourse_end'].shift(1).loc[gap_idxs] + 1
        ends = df_base['discourse_start'].loc[gap_idxs] - 1
        df_mid = pd.DataFrame({
            'discourse_start': starts.values,
            'discourse_end': ends.values,
            'discourse_type': ['Nothing'] * len(gap_idxs),
            'gap_length': [np.nan] * len(gap_idxs),
            'gap_end_length': [np.nan] * len(gap_idxs)
        })
    else:
        df_mid = pd.DataFrame(columns=cols)
    # final gap after last segment
    if df_base['gap_end_length'].iloc[-1] > 0:
        final_start = df_base['discourse_end'].iloc[-1] + 1
        final_end = final_start + df_base['gap_end_length'].iloc[-1]
        df_final = pd.DataFrame({
            'discourse_start': [final_start],
            'discourse_end': [final_end],
            'discourse_type': ['Nothing'],
            'gap_length': [np.nan],
            'gap_end_length': [np.nan]
        })
    else:
        df_final = pd.DataFrame(columns=cols)
    # combine and sort once
    df_out = pd.concat([df_base, df_mid, df_final], ignore_index=True)
    return df_out.sort_values('discourse_start').reset_index(drop=True)


CPU times: user 4 µs, sys: 1e+03 ns, total: 5 µs
Wall time: 7.15 µs


In [4]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_19_try_0.pickle

migration speed (bps): 755231505.5096366
---------------------------
variables to migrate:
sample_submission 569
ax1 536
np 72
IREWR_tmp 914
ax2 536
cols_to_merge 120
IREWR_plug_2 61
i_3 28
add_gap_rows 144
last_ones 106820
plt 72
FuncFormatter 1072
stopwords 48
train_txt 126104
warnings 72
t 166
style 72
myid 61
CountVectorizer 1072
av_per_essay 1841
myword 28
ax 1364
mylen 28
len_dict 589920
all_gaps 2304
word_dict 589920
txt_file 208
data 2813
glob 144
pd 72
test_txt 120
total_gaps 10924
train_first_last 428
orig_output 16
train_first 380
train_last 643
sample_discourse_id 32
train 806064
tqdm 1072
i_1 28
counter 28
cols_to_display 120
IREWR_tmp2 954
IREWR_plug_1 61
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/colinc/code/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_19_try_0.pickle


In [5]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['train', 'train_txt']
Intermediate variables ['test_txt', 'sample_submission']
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables ['train']
Active variables ['sample_discourse_id']
Intermediate variables []
Future variables ['train', 'train_txt']
Modified dataframes
======= Cell 2 =======
Input variables ['train']
Active variables ['train', 'cols_to_display']
Intermediate variables []
Future variables ['train_txt', 'sample_discourse_id']
Modified dataframes
  train
    Input columns: set()
    Changed columns: set()
    Created columns: {'pred_len', 'discourse_len'}
    Deleted columns: set()
======= Cell 3 =======
Input variables ['train', 'cols_to_display']
Active variables []
Intermediate variables []
Future variables ['train', 'train_txt', 'cols_to_display', 'sample_discourse_id']
Modified dataframes
======= Cell 4 =======
Input variables ['train', 'sample_discourse_id']
Active variables []
Inte

In [6]:

with open("/home/colinc/code/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/opt_cell_exec_info_19_try_0.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[19], f)


In [7]:
opt_output = Out.get(4)

In [8]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/annotated_cpu/checkpoints/post_cell_19.pickle

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output


trying: ['sample_submission']
me:  1
trying: ['ax1']
me:  13
trying: ['np']
me:  0
trying: ['IREWR_tmp']
me:  31
trying: ['ax2']
me:  13
trying: ['cols_to_merge']
me:  25
trying: ['IREWR_plug_2']
me:  31
trying: ['i_3']
me:  25
trying: ['last_ones']
me:  25
trying: ['plt']
me:  0
trying: ['FuncFormatter']
me:  0
trying: ['stopwords']
me:  0
trying: ['train_txt']
me:  1
trying: ['warnings']
me:  0
trying: ['t']
me:  23
trying: ['style']
me:  0
trying: ['myid']
me:  23
trying: ['CountVectorizer']
me:  0
trying: ['av_per_essay']
me:  15
trying: ['myword']
me:  23
trying: ['ax']
me:  15
trying: ['mylen']
me:  23
trying: ['len_dict']
me:  23
trying: ['all_gaps']
me:  35
trying: ['word_dict']
me:  23
trying: ['txt_file']
me:  23
trying: ['data']
me:  23
trying: ['glob']
me:  0
trying: ['pd']
me:  0
trying: ['test_txt']
me:  1
trying: ['total_gaps']
me:  37
trying: ['train_first_last']
me:  19
trying: ['orig_output']
me:  2
trying: ['train_first']
me:  19
trying: ['train_last']
me:  19
trying